In [1]:
import os

db_path = r"data/tshirts.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("Old database deleted. Ready for fresh start!")
else:
    print("No existing database found. Ready to create a new one.")

No existing database found. Ready to create a new one.


In [2]:
import sqlite3
import random
import os

# --- DB klasörü ve yolu ---
db_folder = r"data"
os.makedirs(db_folder, exist_ok=True)
db_path = os.path.join(db_folder, "tshirts.db")

# --- DB'ye bağlan ---
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# --- Eski tabloları DROP et ---
cursor.execute("DROP TABLE IF EXISTS discounts")
cursor.execute("DROP TABLE IF EXISTS t_shirts")
conn.commit()

# --- t_shirts tablosunu oluştur ---
cursor.execute("""
CREATE TABLE t_shirts (
    t_shirt_id INTEGER PRIMARY KEY AUTOINCREMENT,
    brand TEXT,
    color TEXT,
    size TEXT,
    price INTEGER CHECK(price BETWEEN 10 AND 50),
    stock_quantity INTEGER,
    UNIQUE(brand, color, size)
)
""")

# --- discounts tablosunu oluştur ---
cursor.execute("""
CREATE TABLE discounts (
    discount_id INTEGER PRIMARY KEY AUTOINCREMENT,
    t_shirt_id INTEGER,
    pct_discount REAL CHECK(pct_discount BETWEEN 0 AND 100),
    FOREIGN KEY (t_shirt_id) REFERENCES t_shirts(t_shirt_id)
)
""")
conn.commit()
print("Tables created successfully!")

Tables created successfully!


In [3]:
brands = ['Van Huesen', 'Levi', 'Nike', 'Adidas']
colors = ['Red', 'Blue', 'Black', 'White']
sizes = ['XS', 'S', 'M', 'L', 'XL']

added = set()
target = len(brands) * len(colors) * len(sizes)  # 80 unique

while len(added) < target:
    b = random.choice(brands)
    c = random.choice(colors)
    s = random.choice(sizes)
    key = (b, c, s)
    if key in added:
        continue
    price = random.randint(10, 50)
    stock = random.randint(10, 100)
    cursor.execute("""
        INSERT INTO t_shirts (brand, color, size, price, stock_quantity)
        VALUES (?, ?, ?, ?, ?)
    """, (b, c, s, price, stock))
    added.add(key)

conn.commit()
print(f"{len(added)} unique t_shirts added!")

80 unique t_shirts added!


In [6]:
import random

# --- discounts tablosunu doldur ---
# t_shirt_id'leri çek
cursor.execute("SELECT t_shirt_id FROM t_shirts")
tshirt_ids = [row[0] for row in cursor.fetchall()]

# Her tshirt için rastgele 0-50% arası discount ekle
for t_id in tshirt_ids:
    pct_discount = random.randint(0, 50)  # yüzde 0-50 arası
    cursor.execute("""
        INSERT INTO discounts (t_shirt_id, pct_discount)
        VALUES (?, ?)
    """, (t_id, pct_discount))

conn.commit()
print(f"{len(tshirt_ids)} discounts added!")

80 discounts added!
